# Chapter 07: Forty-eight messages

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "07"
__file__ = os.path.abspath("run.py")

## The scene

Digital forensics has pulled the messages from Reyes's work phone, 1 to 14 March. Forty-eight of
them. Okafor has already sorted them by hand into three piles: threat, logistics, personal. Varma
does not want his piles. She wants to know whether LANTERN can sort the next phone on its own,
because there will be a next phone. "Show me how often it agrees with Daniel," she says. "And show
me where it doesn't."

The messages are in [`case/messages.jsonl`](../../case/messages.jsonl), each with Okafor's label.
Those labels are the **gold standard** you will score against.

## The concept

The number in "N-shot" is the number of worked examples in the prompt.

| Style | What the prompt contains | When it fits |
|---|---|---|
| **Zero-shot** | The task and the label definitions. No examples. | The task is common and the labels are self-explanatory. |
| **One-shot** | The task plus one worked example. | You need to show the output format, or one tricky convention. |
| **Few-shot** | The task plus several examples (typically 2 to 8). | The labels follow house rules the model cannot guess. |

An example is not an instruction. It is evidence of a pattern, and the model continues patterns.
That makes examples powerful and also dangerous: the model learns everything the examples have in
common, including things you did not mean to teach.

```mermaid
flowchart LR
    subgraph Zero-shot
      Z1[System: task + labels] --> Z2[User: message] --> Z3[Assistant: ?]
    end
    subgraph Few-shot as chat turns
      F1[System: task + labels] --> F2[User: example 1] --> F3[Assistant: personal]
      F3 --> F4[User: example 2] --> F5[Assistant: logistics]
      F5 --> F6[...] --> F7[User: message] --> F8[Assistant: ?]
    end
```

## How it works

This lab sends few-shot examples as **earlier turns of the conversation**: each example is a user
message followed by the "correct" assistant reply. Chat models were trained on conversations, so
this is the shape they continue most reliably. Pasting examples as a block of text inside one user
message also works, but small models are more likely to copy the block's layout (answering
`label: threat` or repeating the message) instead of answering.

![Zero, one and few-shot prompt layouts](../../docs/diagrams/07-shots.svg)

The script runs five prompts over the same 36 messages. Twelve messages are used as examples
somewhere, so all twelve are held out of scoring in every mode. That keeps the comparison fair: no
prompt is ever graded on a message it was shown the answer to.

```mermaid
flowchart TB
    D[48 labelled messages] --> E[12 used as examples:<br/>6 clear, 6 boundary]
    D --> S[36 held out for scoring]
    S --> P1[zero-shot, primed]
    S --> P2[zero-shot]
    S --> P3[one-shot]
    S --> P4[few-shot, clear examples]
    S --> P5[few-shot, boundary only]
    P1 & P2 & P3 & P4 & P5 --> A[Accuracy, label counts,<br/>confusion table, disagreements]
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import time
from collections import Counter

from lantern import case, chat, learn
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

LABELS = ["threat", "logistics", "personal"]

chapter("07", "Forty-eight messages", """
Digital forensics has pulled the messages from Reyes's work phone, 1 to 14 March. Forty-eight of
them. Okafor has already sorted them by hand into three piles: threat, logistics, personal. Varma
does not want his piles. She wants to know whether LANTERN can sort the next phone on its own,
because there will be a next phone. "Show me how often it agrees with Daniel," she says. "And show
me where it doesn't."
""")

messages = case.jsonl("messages.jsonl")
by_id = {m["id"]: m for m in messages}
evidence("messages.jsonl (first 4 of 48)", "\n".join(
    f"{m['id']}  {m['time']}  {m['from']:<20} {m['text']}" for m in messages[:4]))

# Two example sets. Every message used as an example in any mode is held out of scoring.
EASY = ["M03", "M02", "M01", "M12", "M04", "M05"]  # clear cases, two per label, interleaved
BORDER = ["M16", "M24", "M33", "M26", "M15", "M10"]  # judgement calls on the label boundaries
# With --mine, the classifier is yours (practice/exercise.py): it picks the clear examples per
# label from the same six messages, and answers through a schema with one enum field.
MINE = learn.mine()
if MINE:
    EASY = [m["id"] for m in MINE.choose_examples([by_id[i] for i in sorted(EASY)], LABELS,
                                                  per_label=2)]
scored = [m for m in messages if m["id"] not in EASY + BORDER]

TASK = """Classify a text message from a company executive's work phone.
Labels:
- threat: intimidation, pressure or warnings aimed at the recipient
- logistics: work operations, schedules, deliveries, invoices
- personal: family, friends, health, private life
Answer with exactly one word: threat, logistics or personal."""
PRIMED = TASK.replace("a company executive's", "a murdered company executive's")


def ask(m: dict) -> str:
    return f"Message from {m['from']}: \"{m['text']}\""


def examples_as_turns(ids: list[str]) -> list[dict]:
    """Few-shot examples as earlier chat turns: the shape chat models were trained on."""
    turns = []
    for i in ids:
        turns += [{"role": "user", "content": ask(by_id[i])},
                  {"role": "assistant", "content": by_id[i]["label"]}]
    return turns


def parse(text: str) -> str:
    for word in text.lower().replace(".", " ").replace(":", " ").split():
        if word in LABELS:
            return word
    return "invalid"


MODES = {
    "zero-shot, primed": (PRIMED, []),
    "zero-shot": (TASK, []),
    "one-shot": (TASK, ["M16"]),
    "few-shot, clear examples": (TASK, EASY),
    "few-shot, boundary only": (TASK, BORDER),
}

### 1. What the model actually receives in few-shot

In [ ]:
step("1. What the model actually receives in few-shot")
convo = examples_as_turns(EASY[:2]) + [{"role": "user", "content": ask(by_id["M07"])}]
reply("few-shot as chat turns (first 2 of 6 examples)", "```text\n" + "\n".join(
    f"{t['role']:>9}: {t['content']}" for t in [{"role": "system", "content": "(the task)"},
                                                 *convo]) + "\n```", style="blue")
note("Each example is a user turn and the 'right' assistant reply. "
     "The model continues the pattern.")
note("The task is `TASK` in run.py. The example sets are the `EASY` and `BORDER` lists.")

gold = Counter(m["label"] for m in scored)
p_best = learn.predict("Which prompt will agree with Okafor most often?", list(MODES))
p_primed = learn.predict(
    f"Okafor labelled {gold['threat']} of these messages threat. How many will the primed "
    "zero-shot prompt label threat?",
    [f"fewer than {gold['threat'] - 2}", f"{gold['threat'] - 2} to {gold['threat'] + 2}",
     f"more than {gold['threat'] + 2}"],
)

### 2. Five prompts, {len(scored)} held-out messages each

In [ ]:
step(f"2. Five prompts, {len(scored)} held-out messages each")
results: dict[str, dict] = {}
for mode, (system, ids) in MODES.items():
    start = time.perf_counter()
    preds, tokens = [], 0
    if MINE:
        mine_res = MINE.score(scored, chat=chat, system=system,
                              examples=[by_id[i] for i in ids], labels=LABELS)
        preds, tokens = list(mine_res["preds"]), mine_res["prompt_tokens"]
    for m in scored if not MINE else []:
        r = chat([*examples_as_turns(ids), {"role": "user", "content": ask(m)}], system=system,
                 temperature=0, max_tokens=8)
        preds.append(parse(r.text))
        tokens += r.prompt_tokens
    correct = sum(p == m["label"] for p, m in zip(preds, scored, strict=True))
    results[mode] = {"preds": preds, "acc": correct / len(scored), "tokens": tokens,
                     "secs": time.perf_counter() - start, "counts": Counter(preds)}

table = Table(title="Agreement with Okafor's labels (qwen3:1.7b, temperature 0)")
for col in ("prompt", "examples", "accuracy", "threat", "logistics", "personal", "invalid",
            "tokens/msg"):
    table.add_column(col, justify="left" if col == "prompt" else "right")
for mode, res in results.items():
    c = res["counts"]
    table.add_row(mode, str(len(MODES[mode][1])), f"{res['acc']:.0%}", str(c["threat"]),
                  str(c["logistics"]), str(c["personal"]), str(c["invalid"]),
                  str(res["tokens"] // len(scored)))
table.add_row("[dim]Okafor (gold)[/dim]", "", "", str(gold["threat"]), str(gold["logistics"]),
              str(gold["personal"]), "", "")
console.print(table)
note("One word in the task, 'murdered', changes how many messages look like threats.")
note("Examples cost tokens on every call. They earn their place only if accuracy moves.")

best = max(results, key=lambda k: results[k]["acc"])
primed_threats = results["zero-shot, primed"]["counts"]["threat"]
learn.reveal(p_best, answer=list(MODES).index(best),
             why=f"{best} scored {results[best]['acc']:.0%}. Whether examples help, and which "
                 "ones, is an empirical question: score every prompt on the same held-out set.")
learn.reveal(p_primed,
             answer=learn.bucket(primed_threats, [gold["threat"] - 3, gold["threat"] + 2]),
             why=f"{primed_threats} threat labels against Okafor's {gold['threat']}, from one "
                 "word in the task. Everything in a prompt is evidence about the expected answer.")

p_missed = learn.predict(f"Of Okafor's {gold['threat']} threats, how many will the best prompt "
                         "miss?", ["none", "1 or 2", "3 or more"])

### 3. Where the best prompt ({best}) disagrees with Okafor

In [ ]:
step(f"3. Where the best prompt ({best}) disagrees with Okafor")
preds = results[best]["preds"]
confusion = Counter((m["label"], p) for m, p in zip(scored, preds, strict=True))
ct = Table(title="Confusion table (rows: Okafor, columns: LANTERN)")
ct.add_column("gold \\ predicted")
for lab in [*LABELS, "invalid"]:
    ct.add_column(lab, justify="right")
for g in LABELS:
    ct.add_row(g, *[str(confusion[(g, p)]) for p in [*LABELS, "invalid"]])
console.print(ct)
wrong = Table(title="Disagreements")
for col in ("id", "from", "message", "Okafor", "LANTERN"):
    wrong.add_column(col)
for m, p in zip(scored, preds, strict=True):
    if p != m["label"]:
        wrong.add_row(m["id"], m["from"], m["text"], m["label"], p)
console.print(wrong)
missed = confusion[("threat", "logistics")] + confusion[("threat", "personal")] \
    + confusion[("threat", "invalid")]
learn.reveal(p_missed, answer=learn.bucket(missed, [0, 2]),
             why=f"The best prompt missed {missed} of {gold['threat']} threats while scoring "
                 f"{results[best]['acc']:.0%}. A headline accuracy can hide the errors that "
                 "matter most; the confusion table shows them.")
note("Read each disagreement before blaming the model. "
     "Some of Okafor's labels are judgement calls.")
note("M47, Simon Clarke at 21:44 on the night: 'Are you still at the office?' Okafor says threat.")
note("On its own the text is harmless. "
     "It is a threat only if you know what happened two hours later.")

acc = {mode: round(res["acc"], 4) for mode, res in results.items()}
per_msg = {mode: res["tokens"] // len(scored) for mode, res in results.items()}
learn.record(
    acc_zero_primed=acc["zero-shot, primed"],
    acc_zero=acc["zero-shot"],
    acc_one=acc["one-shot"],
    acc_few_clear=acc["few-shot, clear examples"],
    acc_few_border=acc["few-shot, boundary only"],
    best_accuracy=acc[best],
    few_beats_zero=acc["few-shot, clear examples"] > acc["zero-shot"],
    threats_gold=gold["threat"],
    threats_zero_primed=primed_threats,
    threats_zero=results["zero-shot"]["counts"]["threat"],
    threats_few_clear=results["few-shot, clear examples"]["counts"]["threat"],
    threats_missed_best=missed,
    invalid_total=sum(res["counts"]["invalid"] for res in results.values()),
    tokens_zero=per_msg["zero-shot"],
    tokens_few_clear=per_msg["few-shot, clear examples"],
    scored_messages=len(scored),
    mine=bool(MINE),
)

your_turn(
    "07",
    check=[
        "Compare each prompt's threat column with Okafor's gold row. Which prompts count fewer "
        "threats than he did, even with a good accuracy score?",
        "In the Disagreements table, who sent the threats LANTERN missed? Open case/messages.jsonl "
        "and read that sender's other messages.",
        "Compare tokens/msg with accuracy. Did six examples pay for what they cost on every call?",
    ],
    exercises=[
        "In run.py, reorder `EASY` so the two threat examples, M03 and M12, come last: "
        "[\"M02\", \"M01\", \"M04\", \"M05\", \"M03\", \"M12\"]. Does the threat count rise?",
        "In run.py, add \"M07\" and \"M23\" to `EASY`. They drop out of `scored` on their own. "
        "Do the other +88 301 messages now land as threats?",
        "In run.py, step 2, replace the `examples_as_turns(ids)` turns with one user message: each "
        "example as a text line (message, then label), then ask(m). Count the invalid column.",
        "No edit: run LANTERN_MODEL=qwen3:4b make lesson N=07. Does 'few-shot, boundary only' "
        "still collapse?",
    ],
)

finding(f"""
Best prompt on this phone: {best}, {results[best]['acc']:.0%} agreement with Okafor. The threats
come from one number, +88 301, the same prefix as Northwind's bank in Port Sabran. They start on
1 March and end at 15:03 on 14 March with a place and a time: "Stairwell C. 23:30."

Four more messages Okafor filed as threats are not from that number. Two are from Raj Malhotra.
Two are from Simon Clarke.
""")
save_shot("07-zero-one-few-shot")

## The nuances

**1. Measure, do not assume.** Few-shot is often described as "better". On this phone, with this
model, in the captured run, the clear-example few-shot prompt scored 83%, exactly what zero-shot
scored, at three times the tokens per message. (An earlier run had it three points ahead.) The only way to know whether examples help is to score both on held-out data.

**2. The framing is an example too.** The "primed" prompt differs from the plain zero-shot prompt by
one word: "a *murdered* company executive's work phone". That word took the model from 4 threat
labels to 26, and accuracy from 83% to 50%. Everything in the prompt is evidence about what answer
is expected. Keep task descriptions neutral.

**3. Label balance leaks.** If four of your six examples are threats, the model learns that threats
are common. Use roughly the label mix you expect in real data, or an even mix if you do not know.

**4. Order matters, and the last example pulls hardest.** Models weight recent context heavily
(**recency bias**). If all the threat examples come last, borderline messages tilt toward threat.
Interleave labels and end on the least dramatic one.

**5. Examples set the format.** The model copies the shape of the example answers: one word, a
word with a full stop, a sentence. If the examples are sloppy, the output is sloppy. If they answer
in one lowercase word, it will too.

**6. Boundary examples need clear examples around them.** "Boundary only" used six judgement calls
(a colleague's pressure counted as threat, a frightened wife counted as personal, a CEO's probing
question counted as logistics). A large model can learn the rule from those. The 1.7B model learned
"everything is logistics" (0 threats, 29 logistics) and fell to 64%. Teach the clear cases first,
then the edges.

**7. One example can be worse than none.** One-shot with a single threat example (Malhotra's "Can we
talk before you take anything to Elena?") scored 78%, below zero-shot. A single example is a strong
signal about one corner of the space and nothing about the rest.

**8. Gold labels are opinions.** Okafor marked M47, Clarke's "Are you still at the office?", as a
threat. On its own the text is harmless. It is a threat only if you know what happened two hours
later. When the model "gets it wrong", read the message before blaming the model. Some
disagreements are about context the model was never given.

**9. Examples cost tokens on every call.** Six examples took each call from about 105 prompt tokens
to about 320. On a phone with 10,000 messages that is the difference between minutes and hours on a
laptop, and real money on a paid API.

## On a hosted model

```bash
make lesson N=07 TIER=hosted              # needs ANTHROPIC_API_KEY in .env, uv sync --extra cloud
make lesson N=07 TIER=hosted ARGS=--mine  # your classifier, with native structured outputs
make lesson N=07 TIER=hosted REPLAY=1     # once a hosted run is recorded (RECORD=1), no key
```

The lesson's own path asks for one word and parses it, on both tiers. Your classifier from the
exercise (`--mine`) sends the label schema as `format=`, which the hosted tier turns into
Claude's native structured outputs (`output_config` with a JSON schema), so every answer is one
of the three labels. Claude takes no `temperature`: the lesson's `temperature=0` is reported and
not sent (see `src/lantern/cloud.py`), so the hosted scores are at the provider's own sampling.

| Measure | Local: qwen3:1.7b | Hosted: claude-haiku-5-5 |
|---|---|---|
| Accuracy: zero-shot, primed | 50% | not run |
| Accuracy: zero-shot | 83% | not run |
| Accuracy: one-shot | 78% | not run |
| Accuracy: few-shot, clear examples | 83% | not run |
| Accuracy: few-shot, boundary only | 64% | not run |
| Threat labels, primed (gold: 8) | 26 | not run |
| Prompt tokens per message, zero-shot / six examples | 105 / 319 | not run |

Local numbers are from the run in "What you should see" (qwen3:1.7b, local tier, Apple M1 Pro 16 GB, temperature 0).

What the hosted run will test: the collapse of "boundary only" to almost all logistics is
expected to be a small-model effect, since a larger model can infer the rule from edge cases;
the pull of the word "murdered" toward threat, and the cost of examples on every call, should
hold on both. M47 should stay a disagreement on any model: the text does not carry the context.

## Try this

1. Swap the order of `EASY` so both threat examples come last. Does the threat count rise?
2. Add `M07` and `M23` to the clear examples (and so to the held-out list). Do the remaining
   +88 301 messages now land as threats?
3. Put the six examples inside the user message as text instead of chat turns. Count the
   `invalid` answers.
4. Run with `LANTERN_MODEL=qwen3:4b make lesson N=07`. Does "boundary only" still collapse?

## Master it

After this chapter you can build zero, one and few-shot prompts with examples as chat turns,
choose and order examples per label, constrain a classifier's answer with a schema, and read a
confusion table instead of trusting one accuracy figure.

**Predict, then see.** `make lesson N=07` stops three times to ask which prompt agrees with
Okafor most, how many threats the primed prompt finds against his count, and how many of his
threats the best prompt misses. Each answer is checked against your own run. Add
`ARGS=--no-predict` to watch without stopping.

**Build it.** Write the few-shot classifier in [`practice/exercise.py`](practice/exercise.py):
a label schema, examples chosen per label and interleaved, examples as turns, and a scorer.

```bash
make exercise N=07             # 8 offline tests, scripted model, no Ollama needed
make hint N=07                 # one hint at a time
make solution N=07             # the reference, once you have tried
make lesson N=07 ARGS=--mine   # all five prompts through your classifier
```

**Check yourself.** `make quiz N=07`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=07` checks your latest run against three targets: your
classifier with no invalid answers and at least 80% on clear examples; a reworded task that
finds twice Okafor's threats; and clear examples at 220 tokens a message or fewer without
losing agreement.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 07

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 07

In [ ]:
%run ../../scripts/learn.py hint 07

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 07

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The threats come from one number, +88 301, the same prefix as Northwind's bank in Port Sabran. They
start on 1 March and end at 15:03 on 14 March with a place and a time: "Stairwell C. 23:30."

Four more messages Okafor filed as threats are not from that number. Two are from Raj Malhotra. Two
are from Simon Clarke.

**Next:** [Chapter 08: Rows for the link chart](../08-structured-output/README.md)